In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Physical and tight-binding parameters
a = 1.0          # Lattice constant
t = 1.0          # Nearest-neighbor hopping magnitude
t2_ratio = 0.25  # Next-nearest-neighbor hopping ratio (t/4) in y-direction
epsilon = 0.0    # On-site energy

In [ ]:
# Hermitian conjugate helper
dagger = lambda M: M.conj().T

def Hcell(eps, t1):
    """Constructs the 2x2 intra-cell block H0."""
    return np.asarray([
        [eps, t1],
        [np.conjugate(t1), eps]
    ], dtype=complex)

def T(t1, t2):
    """Constructs the 2x2 inter-cell coupling block T0."""
    return np.asarray([
        [t2, 0.0],
        [np.conjugate(t1), t2]
    ], dtype=complex)

def build_ribbon_Hamiltonian(kx, M, eps, t_val, t2_val, lat_a):
    """
    Constructs the M x M tight-binding Hamiltonian H(kx) for an arbitrary ribbon width M,
    automatically filling zeros for non-neighboring blocks and supporting both even and odd M.
    """
    # k-dependent intra-row hopping t1(kx)
    t1 = -t_val * (1.0 + np.exp(-1j * kx * np.sqrt(2) * lat_a))
    t2 = t_val * t2_val
    
    H0 = Hcell(eps, t1)
    T0 = T(t1, t2)
    
    n_blocks = int(np.ceil(M / 2))
    H = H0.copy()
    
    # Iteratively build block-tridiagonal matrix
    for _ in range(1, n_blocks):
        zeros_pad = np.zeros((H.shape[0] - 2, 2), dtype=complex)
        col_block = np.vstack([zeros_pad, T0])
        
        H = np.block([
            [H,                 col_block],
            [dagger(col_block), H0       ]
        ])
        
    # Slice to exact width M (handles odd M by trimming the final B site)
    return H[:M, :M]

: 

In [ ]:
# Brillouin zone boundary for 1D unit cell of length sqrt(2)a
kx_max = np.pi / (np.sqrt(2) * a)
kx_grid = np.linspace(-kx_max, kx_max, 401)

# Choose ribbon widths: W1 (even/odd comparison) and W2 ≈ 2*W1
widths = [16, 17, 32, 33] 

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)

for ax, M in zip(axes.flat, widths):
    # Solve H(kx) across kx_grid using vectorized evaluation
    H_stack = np.array([build_ribbon_Hamiltonian(kx, M, epsilon, t, t2_ratio, a) for kx in kx_grid])
    energies, _ = np.linalg.eigh(H_stack)  # shape: (len(kx_grid), M)
    
    ax.plot(kx_grid / kx_max, energies / t, color='navy', lw=1.0)
    
    parity_str = "Even M (Distinct Edges)" if M % 2 == 0 else "Odd M (Symmetric Edges)"
    ax.set_title(f"Ribbon Width $M = {M}$ ({parity_str})", fontsize=11)
    ax.set_xlabel(r"$k_x / k_{x,\text{max}}$", fontsize=10)
    ax.set_ylabel(r"$E_n(k_x) / t$", fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle("Task 1: Ribbon Band Structures $E_n(k_x)$", fontsize=14, y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
# Select a wide ribbon to clearly resolve bulk and edge profiles without finite-size hybridization
M_wide = 41
H_stack_wide = np.array([build_ribbon_Hamiltonian(kx, M_wide, epsilon, t, t2_ratio, a) for kx in kx_grid])
energies_wide, states_wide = np.linalg.eigh(H_stack_wide)  # states shape: (len(kx_grid), M, M)

y_sites = np.arange(1, M_wide + 1)

# Pick representative states (ik_index, band_index, description)
# Note: states[ik, :, n] gives the eigenvector psi_{nk} for band n at k-index ik
selection_indices = [
    (200, 0,  r"Center of BZ ($k_x = 0$), Lowest Band $n=1$ (Bulk Standing Wave)"),
    (200, 20, r"Center of BZ ($k_x = 0$), Mid-Spectrum Band $n=21$"),
    (360, 0,  r"Off-Center ($k_x = 0.8 k_{\text{max}}$), Band $n=1$ (Spatial Beating)"),
    (396, 0,  r"Near BZ Edge ($k_x \approx k_{\text{max}}$), Band $n=1$ (Sublattice Polarization)")
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (ik, n, desc) in zip(axes.flat, selection_indices):
    prob_density = np.abs(states_wide[ik, :, n])**2
    
    ax.plot(y_sites, prob_density, '-o', color='crimson', markersize=4, lw=1.2)
    ax.set_title(desc, fontsize=10)
    ax.set_xlabel("Transverse site index $i$ ($y_i$)", fontsize=9)
    ax.set_ylabel(r"$|\psi_{nk}(i)|^2$", fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Task 2: Transverse Wavefunction Profiles $|\psi_{nk}(i)|^2$", fontsize=14, y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
# Explore Task 3: Modifying next-nearest-neighbor y-hopping ratio or adding edge potentials
t2_altered = 0.5  # Doubled next-nearest neighbor hopping in y-direction

H_stack_altered = np.array([build_ribbon_Hamiltonian(kx, M_wide, epsilon, t, t2_altered, a) for kx in kx_grid])
energies_altered, _ = np.linalg.eigh(H_stack_altered)

plt.figure(figsize=(8, 5))
plt.plot(kx_grid / kx_max, energies_altered / t, color='darkgreen', lw=1.0)
plt.title(f"Task 3: Band Structure with Altered Anisotropy ($t_{{2,y}} = 0.5t$)", fontsize=11)
plt.xlabel(r"$k_x / k_{x,\text{max}}$", fontsize=10)
plt.ylabel(r"$E_n(k_x) / t$", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()